# Análise exploratória: Indicador Criança Alfabetizada

Tech Challenge Fase 3 · Pós-Graduação AI Scientist (FIAP)

Este notebook percorre a base analítica construída no passo anterior e levanta as
evidências que orientam a modelagem. As conclusões consolidadas estão em
[`reports/analise-exploratoria.md`](../reports/analise-exploratoria.md).

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import matplotlib.pyplot as plt

from config import settings

df = pd.read_parquet("../data/processed/base_analitica.parquet")
print(f"{len(df):,} alunos | {df['id_municipio'].nunique():,} municípios | "
      f"{df['id_escola'].nunique():,} escolas | anos {sorted(df['ano'].unique())}")

3,867,999 alunos | 5,548 municípios | 42,811 escolas | anos [np.int64(2023), np.int64(2024)]


## 1. Variáveis que determinam o alvo

Antes de escolher preditores é preciso saber o que não pode entrar. A primeira
suspeita é a `proficiencia`, já que o alvo vem de um corte sobre ela.

In [2]:
corte = settings.PONTO_CORTE_SAEB
fez = df[df["proficiencia"].notna()]

concordancia = ((fez["proficiencia"] >= corte).astype(int) == fez["alfabetizado"]).mean()

print(f"proficiência máxima entre não alfabetizados: {fez.loc[fez.alfabetizado==0,'proficiencia'].max():.6f}")
print(f"proficiência mínima entre alfabetizados:     {fez.loc[fez.alfabetizado==1,'proficiencia'].min():.6f}")
print(f"concordância (proficiencia >= {corte}) == alvo: {100*concordancia:.4f}%")

proficiência máxima entre não alfabetizados: 742.999819
proficiência mínima entre alfabetizados:     743.000000
concordância (proficiencia >= 743) == alvo: 100.0000%


Concordância perfeita. A variável não prediz o alvo: ela é o alvo em escala
contínua.

### Outras colunas com o mesmo comportamento

O cruzamento com as demais revelou algo que eu não esperava.

In [3]:
for col in ["presenca", "preenchimento_caderno"]:
    print(f"\n{col} x alfabetizado (proporção por linha):")
    print(pd.crosstab(df[col], df["alfabetizado"], normalize="index").round(4))


presenca x alfabetizado (proporção por linha):


alfabetizado       0       1
presenca                    
0             1.0000  0.0000
1             0.4086  0.5914

preenchimento_caderno x alfabetizado (proporção por linha):
alfabetizado                0       1
preenchimento_caderno                
0                      1.0000  0.0000
1                      0.4084  0.5916


Quem tem `presenca = 0` é classificado como não alfabetizado em 100% dos casos, e
o mesmo vale para `preenchimento_caderno`. Ausência é registrada como não
alfabetização. As quatro colunas ficam fora do treino, incluindo `peso_aluno`, que
é atribuído depois da prova.

In [4]:
print("Excluídas por vazamento:", settings.COLUNAS_VAZAMENTO)

Excluídas por vazamento: ['proficiencia', 'peso_aluno', 'preenchimento_caderno', 'presenca']


## 2. Composição da classe negativa

Se ausência vira rótulo negativo, parte da classe 0 não é desempenho ruim, e sim
falta de medição. Quanto?

In [5]:
zeros = df[df["alfabetizado"] == 0]
ausentes = zeros["proficiencia"].isna().sum()

print(f"não alfabetizados .............. {len(zeros):,}")
print(f"  fizeram e não atingiram ...... {len(zeros)-ausentes:,}")
print(f"  ausentes, sem prova .......... {ausentes:,}  ({100*ausentes/len(zeros):.1f}%)")

não alfabetizados .............. 1,883,453
  fizeram e não atingiram ...... 1,370,115
  ausentes, sem prova .......... 513,338  (27.3%)


Mais de um quarto da classe negativa nunca foi avaliado. Como as variáveis que
sinalizam ausência são as excluídas por vazamento, esses alunos entrariam no
treino sem sinal aprendível. A modelagem usa a população testada.

## 3. Efeito da ausência sobre o indicador regional

O indicador oficial soma os dois efeitos. Separá-los reordena as regiões.

In [6]:
comp = pd.DataFrame({
    "oficial":  df.groupby("regiao")["alfabetizado"].mean().mul(100),
    "testados": fez.groupby("regiao")["alfabetizado"].mean().mul(100),
    "ausencia": df.groupby("regiao")["proficiencia"].apply(lambda s: 100*s.isna().mean()),
}).round(1).sort_values("testados", ascending=False)

comp["salto"] = (comp["testados"] - comp["oficial"]).round(1)
comp

,oficial,testados,ausencia,salto
regiao,,,,
Sul,52.5,64.5,18.5,12.0
Centro-Oeste,54.7,62.0,11.8,7.3
Sudeste,53.6,61.3,12.6,7.7
Nordeste,50.6,55.8,9.2,5.2
Norte,41.8,50.9,18.0,9.1


O Sul tem o melhor desempenho entre quem faz a prova, mas cai no indicador
oficial por ter a maior ausência do país. O Nordeste é o inverso: desempenho
abaixo da média com a maior participação.

São dois problemas com respostas de política pública distintas.

## 4. Origem do sinal preditivo

In [7]:
cols = ["taxa_municipio", "meta_municipio", "gap_municipio", "taxa_uf"]
fez[cols + ["alfabetizado"]].corr()["alfabetizado"].drop("alfabetizado").sort_values(ascending=False).round(3)

taxa_municipio    0.326
meta_municipio    0.251
taxa_uf           0.243
gap_municipio    -0.202
Name: alfabetizado, dtype: float64

A taxa do próprio município é a variável mais associada ao desfecho. Do aluno
sobra pouco: `serie` é constante e `rede` tem três valores.

In [8]:
print("valores distintos por coluna do aluno:")
for c in ["serie", "rede", "caderno"]:
    print(f"  {c:10s} {df[c].nunique()}")

valores distintos por coluna do aluno:
  serie      1
  rede       3
  caderno    22


## 5. Cobertura das variáveis de contexto

In [9]:
cobertura = (100*df.notna().mean()).round(2).sort_values()
cobertura[cobertura < 100]

gap_uf             0.00
meta_uf            0.00
gap_municipio     45.93
meta_municipio    45.93
atingiu_meta      46.56
proficiencia      86.73
peso_aluno        86.73
taxa_uf           99.28
taxa_municipio    99.89
rede_desc         99.89
dtype: float64

As metas por UF vieram vazias. Achei que fosse erro do join e fui conferir a Gold
direto no BigQuery: a meta por UF só existe para a rede pública agregada
(`rede = 5`) e apenas em 2024. Nenhum aluno pertence a essa rede, então o join
nunca casa. As duas colunas saem da base.

A `meta_municipio` cobre apenas 2024 na rede municipal. A ausência é sistemática,
não aleatória: informa ano e rede em vez de indicar falha de coleta.

In [10]:
df.groupby(["ano","rede"]).agg(
    alunos=("alfabetizado","size"),
    sem_meta_pct=("meta_municipio", lambda s: round(100*s.isna().mean(),1)),
)

alunos  sem_meta_pct
ano  rede                       
2023 2      155140         100.0
     3     1592299         100.0
2024 2      280258         100.0
     3     1840277           3.5
     4          25         100.0

## 6. O identificador do aluno

Eu pretendia agrupar a validação por aluno até verificar o comportamento do
`id_aluno`.

In [11]:
multi = df.groupby("id_aluno").agg(anos=("ano","nunique"), munis=("id_municipio","nunique"))
rep = multi[multi["anos"] > 1]

print(f"ids presentes nos dois anos ............. {len(rep):,}")
print(f"  destes, com mais de um município ..... {(rep['munis']>1).sum():,} "
      f"({100*(rep['munis']>1).mean():.0f}%)")

ids presentes nos dois anos ............. 1,515,671
  destes, com mais de um município ..... 1,274,442 (84%)


O código é reaproveitado entre anos e não acompanha a mesma criança. Vale lembrar
que toda a base é de 2º ano, então um aluno avaliado em 2023 estaria no 3º ano em
2024 e não deveria reaparecer. Isso descarta features de trajetória individual e
leva o agrupamento da validação para o nível de município.

## 7. O que a modelagem herda

| Decisão | Motivo |
|---|---|
| Excluir `proficiencia`, `presenca`, `preenchimento_caderno`, `peso_aluno` | determinam o alvo |
| Excluir `serie`, `meta_uf`, `gap_uf` | constante e 100% nulas |
| Restringir à população testada | ausência é outro fenômeno |
| Agrupar a validação por município | evita o mesmo território nos dois lados |
| Marcar a ausência de `meta_municipio` | falta sistemática, não aleatória |

**Expectativa realista:** as distribuições de contexto se sobrepõem bastante. O
modelo deve ser informativo, sem pretensão de acerto individual.